# Libraries

In [2]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import requests
import time
from datetime import datetime, timedelta

# Get Billboard Top 100

In [4]:
def scrape_billboard(date_str):
    url = f"https://www.billboard.com/charts/hot-100/{date_str}"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed to load Billboard for {date_str}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    songs = []

    # Each song entry is inside this list item
    entries = soup.select("li.o-chart-results-list__item")

    for entry in entries:
        title_elem = entry.select_one("h3.c-title")
        artist_elem = entry.select_one("span.c-label")

        if title_elem and artist_elem:
            title = title_elem.get_text(strip=True)
            artist = artist_elem.get_text(strip=True)

            # Filter out ranks, NEW, RE-ENTRY, etc.
            if title and artist and not artist[0].isdigit() and artist.upper() != "NEW":
                songs.append((title, artist))

    return songs

# --- Set your start and end dates here ---
start_date = datetime(2010, 1, 1)
end_date = datetime(2020, 1, 1)

# --- Generate all weekly dates between start and end ---
dates = []
current = start_date
while current <= end_date:
    dates.append(current.strftime("%Y-%m-%d"))
    current += timedelta(weeks=1)

# --- Scrape Billboard for each week ---
billboard_data = {}  # Billboard songs per date
unique_songs = set()  # Deduplicated (title, artist) pairs

for date in dates:
    print(f"📅 Scraping Billboard for {date}...")
    chart = scrape_billboard(date)
    billboard_data[date] = chart
    for title, artist in chart:
        unique_songs.add((title, artist))  # Deduplicate

# --- Final Output ---
print(f"\n🎧 Total unique songs across {len(dates)} weeks in the 2010s: {len(unique_songs)}")

📅 Scraping Billboard for 2010-01-01...
📅 Scraping Billboard for 2010-01-08...
📅 Scraping Billboard for 2010-01-15...
📅 Scraping Billboard for 2010-01-22...
📅 Scraping Billboard for 2010-01-29...
📅 Scraping Billboard for 2010-02-05...
📅 Scraping Billboard for 2010-02-12...
📅 Scraping Billboard for 2010-02-19...
📅 Scraping Billboard for 2010-02-26...
📅 Scraping Billboard for 2010-03-05...
📅 Scraping Billboard for 2010-03-12...
📅 Scraping Billboard for 2010-03-19...
📅 Scraping Billboard for 2010-03-26...
📅 Scraping Billboard for 2010-04-02...
📅 Scraping Billboard for 2010-04-09...
📅 Scraping Billboard for 2010-04-16...
📅 Scraping Billboard for 2010-04-23...
📅 Scraping Billboard for 2010-04-30...
📅 Scraping Billboard for 2010-05-07...
📅 Scraping Billboard for 2010-05-14...
📅 Scraping Billboard for 2010-05-21...
📅 Scraping Billboard for 2010-05-28...
📅 Scraping Billboard for 2010-06-04...
📅 Scraping Billboard for 2010-06-11...
📅 Scraping Billboard for 2010-06-18...
📅 Scraping Billboard for 

# Get MBIDs of each unique song using the MusicBrainz API

In [6]:
import re

def clean_artist_name_variants(artist):
    variants = set()

    # Original version
    variants.add(artist.strip())

    # Common substitutions
    clean = re.sub(r"\b(featuring|feat\.|with|ft\.|and)\b", "&", artist, flags=re.IGNORECASE)
    clean = re.sub(r"\s*&\s*", " & ", clean).strip()
    clean = re.sub(r"\s{2,}", " ", clean)
    variants.add(clean)

    # Remove collaborators (just main artist)
    main_only = re.split(r"\b(featuring|feat\.|with|ft\.|and|&)\b", artist, flags=re.IGNORECASE)[0].strip()
    variants.add(main_only)

    return list(variants)

In [7]:
def get_mbid(title, artist):
    name_variants = clean_artist_name_variants(artist)

    url = "https://musicbrainz.org/ws/2/recording/"
    headers = {
        'User-Agent': 'BillboardMBIDFetcher/1.0 (your-email@example.com)'
    }

    for name in name_variants:
        query = f'"{title}" AND artist:"{name}"'
        params = {
            'query': query,
            'fmt': 'json',
            'limit': 1
        }
        try:
            response = requests.get(url, params=params, headers=headers)
            data = response.json()
            recordings = data.get("recordings", [])
            if recordings:
                # print(f"✅ Found MBID for '{title}' using artist variant: '{name}'")
                return recordings[0]["id"]
        except Exception as e:
            print(f"❌ Error while searching with artist '{name}': {e}")

    return None

## Loop through unique song list:

In [9]:
billboard_with_mbid = []

for title, artist in unique_songs:
    print(f"🔍 {title} — {artist}")
    mbid = get_mbid(title, artist)
    if mbid:
        billboard_with_mbid.append({
            "title": title,
            "artist": artist,
            "mbid": mbid
        })
    else:
        print(f"⚠️ No MBID found for: {title} — {artist}")
    time.sleep(1)

print(f"\n🎧 Total unique song MBIDs across all weeks: {len(billboard_with_mbid)}")

🔍 Bake Sale — Wiz Khalifa Featuring Travi$ Scott
🔍 Belly — Lil Baby & Gunna
🔍 Circle The Drain — Katy Perry
🔍 Bitch, Don't Kill My Vibe — Kendrick Lamar
🔍 On Top Of The World — Imagine Dragons
🔍 On Everything — DJ Khaled Featuring Travis Scott, Rick Ross & Big Sean
🔍 Get Lucky — Daft Punk Featuring Pharrell Williams
🔍 Get It Started — Pitbull Featuring Shakira
🔍 Hey Girl — Billy Currington
🔍 The Greatest Show — Hugh Jackman, Keala Settle, Zac Efron, Zendaya & The Greatest Showman Ensemble
🔍 Take Back The Night — Justin Timberlake
🔍 Redbone — Childish Gambino
🔍 Losing My Religion — Glee Cast
🔍 Heartless — Dia Frampton
🔍 Remedy — Adele
🔍 Mama's Broken Heart — Miranda Lambert
🔍 Heat — Chris Brown Featuring Gunna
🔍 Singles You Up — Jordan Davis
🔍 Better Than I Know Myself — Adam Lambert
🔍 Don't Judge Me — Chris Brown
🔍 How Not To — Dan + Shay
🔍 Play The Guitar — B.o.B Featuring Andre 3000
🔍 Lord Above — Fat Joe & Dre Featuring Eminem & Mary J. Blige
🔍 I Cry — Flo Rida
🔍 Women Lie, Men Lie 

#  Code to get all the features from AcousticBrainz API

## "AcousticBrainz was a joint effort between **Music Technology Group at Universitat Pompeu Fabra** in Barcelona and the MusicBrainz project. At the heart of this project lies the **Essentia toolkit** from the MTG – this open source toolkit enables the automatic analysis of music. The output from Essentia is collected by the AcousticBrainz project and made available to the public."

### https://essentia.upf.edu/

### https://acousticbrainz.org/data

### "The low-level data includes acoustic descriptors characterizing overall loudness, dynamics and spectral shape of a signal, rhythm descriptors (including beats per minute), and tonal information (including keys and scales). The low-level data is stable, compared to the higher-level data. This hopefully means that we will not have to recompute this data from audio files too frequently."

### "The high-level data includes information about moods, genres, vocals and music type automatically inferred from low-level data by pre-trained classifiers. The high-level data can be computed from the low-level data, without requiring access to original audio files. The high-level data is less stable than the low-level data, since it contains more experimental algorithms."

In [16]:
def get_all_acousticbrainz_features(mbid):
    base_url = "https://acousticbrainz.org/api/v1"
    features = {}

    try:
        # High-level
        hl_url = f"{base_url}/{mbid}/high-level"
        hl_response = requests.get(hl_url)
        if hl_response.status_code == 200:
            features["highlevel"] = hl_response.json()

        # Low-level
        ll_url = f"{base_url}/{mbid}/low-level"
        ll_response = requests.get(ll_url)
        if ll_response.status_code == 200:
            features["lowlevel"] = ll_response.json()

    except Exception as e:
        print(f"❌ Error fetching features for {mbid}: {e}")

    return features if features else None

## Loop through MBIDs:

In [18]:
billboard_with_all_features = []

for track in billboard_with_mbid:
    mbid = track["mbid"]
    print(f"🎧 Fetching full features for: {track['title']} — {track['artist']}")
    features = get_all_acousticbrainz_features(mbid)
    if features:
        track["features"] = features
        billboard_with_all_features.append(track)
    else:
        print(f"⚠️ No features found for MBID: {mbid}")
    time.sleep(1)  # Respect API limits
print(f"\n🎧 Total unique song features across all weeks during the 2010s: {len(billboard_with_all_features)}")

🎧 Fetching full features for: Bake Sale — Wiz Khalifa Featuring Travi$ Scott
⚠️ No features found for MBID: 435bb262-f017-44d6-882e-d9c8dec08cca
🎧 Fetching full features for: Belly — Lil Baby & Gunna
⚠️ No features found for MBID: f9f9f017-e425-47e3-9934-61244f2183c1
🎧 Fetching full features for: Circle The Drain — Katy Perry
🎧 Fetching full features for: Bitch, Don't Kill My Vibe — Kendrick Lamar
⚠️ No features found for MBID: 408a7477-5db2-45cf-af12-a4ae5c085f02
🎧 Fetching full features for: On Top Of The World — Imagine Dragons
⚠️ No features found for MBID: c0d3e4ab-33b7-43b2-a514-ff35bde5ce0e
🎧 Fetching full features for: On Everything — DJ Khaled Featuring Travis Scott, Rick Ross & Big Sean
🎧 Fetching full features for: Get Lucky — Daft Punk Featuring Pharrell Williams
🎧 Fetching full features for: Get It Started — Pitbull Featuring Shakira
⚠️ No features found for MBID: 21984d9d-abd9-4dc1-8112-bac4d749cca9
🎧 Fetching full features for: Hey Girl — Billy Currington
🎧 Fetching full

# Save as a json file

In [20]:
import json

with open("billboard_2010s_all_features.json", "w") as f:
    json.dump(billboard_with_all_features, f, indent=2)

In [21]:
def flatten_dict(d, parent_key='', sep='.'):
    # Recursively flattens a nested dictionary
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def flatten_track_features(track):
    flat = {
        "title": track["title"],
        "artist": track["artist"],
        "mbid": track["mbid"]
    }

    features = track.get("features", {})

    # Flatten high-level and low-level dictionaries
    highlevel_flat = flatten_dict(features.get("highlevel", {}), parent_key="hl")
    lowlevel_flat = flatten_dict(features.get("lowlevel", {}), parent_key="ll")

    # Merge all into one flat dict
    flat.update(highlevel_flat)
    flat.update(lowlevel_flat)
    return flat

# Flatten all songs
rows = [flatten_track_features(track) for track in billboard_with_all_features]

# Create DataFrame and export to CSV
df = pd.DataFrame(rows)
df.to_csv("billboard_2010s_features_full.csv", index=False)

print("✅ Full feature set exported to 'billboard_2010s_features_full.csv'")


✅ Full feature set exported to 'billboard_2010s_features_full.csv'
